In [1]:
# ============================================================
# AMP Challenge — STEP 12
# Final Multi-Objective Selection
# 60K -> Final 50K Library -> Final Top100
# Outputs:
#   - Full CSV 50K
#   - Full CSV Top100
#   - Summary CSV 50K
#   - Summary CSV Top100
#   - FASTA 50K
#   - FASTA Top100
#   - Manifest JSON
# ============================================================

!pip -q install pandas numpy biopython

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import files
from Bio import pairwise2
from Bio.Align import substitution_matrices


# ============================================================
# 0. CONFIG
# ============================================================

OUT_DIR = Path("/content/STEP12_FINAL_SELECTION")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LIBRARY_N = 50000
TOP_N = 100

# Baseline-inspired internal diversity threshold.
# NOT claimed to be the hidden official aggregation formula.
LOCAL_ALIGNMENT_SIM_THRESHOLD = 0.40

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)


# ============================================================
# 1. UPLOAD STEP 11 CSV
# ============================================================

print("Upload:")
print("STEP11_60K_WITH_DIVERSITY_DIAGNOSTICS.csv")

uploaded = files.upload()
input_file = next(iter(uploaded))

df = pd.read_csv(
    input_file,
    low_memory=False
)

df["sequence"] = (
    df["sequence"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("\nInput file:", input_file)
print("Rows:", len(df))
print("Unique sequences:", df["sequence"].nunique())


# ============================================================
# 2. STRICT INPUT VALIDATION
# ============================================================

assert len(df) == 60000, (
    f"Expected 60,000 rows, found {len(df):,}"
)

assert df["sequence"].nunique() == 60000, (
    "Duplicate sequences detected."
)

allowed = set("ACDEFGHIKLMNPQRSTVWY")

invalid = df["sequence"].map(
    lambda s: (
        len(s) < 8
        or len(s) > 50
        or not set(s).issubset(allowed)
    )
)

assert invalid.sum() == 0, (
    f"Invalid sequences found: {invalid.sum()}"
)


# ============================================================
# 3. REQUIRED COLUMNS
# ============================================================

required = [
    "sequence",

    # potency: lower MIC = better
    "apex_mean_mic_uM",

    # safety: higher HC50 = better
    "hemopi2_hc50_uM",

    # physicochemical realism
    "APD6_physchem_realism_diagnostic",

    # novelty: higher = more novel
    "external_novelty_diagnostic",

    # biologically informed embedding
    "esm2_potent_mean_top5_cosine",

    # synthesizability: higher = better
    "pepsysco_score",

    # embedding-space density diagnostic
    "embedding_nn1_cosine_similarity"
]

missing_required = [
    c for c in required
    if c not in df.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}"
    )

print("\nAll required scoring columns found.")


# ============================================================
# 4. CHECK MISSING VALUES
# ============================================================

for col in required[1:]:

    n_missing = int(
        df[col].isna().sum()
    )

    print(
        f"{col}: missing = {n_missing:,}"
    )

    if n_missing > 0:
        raise ValueError(
            f"Missing values in {col}"
        )


# ============================================================
# 5. PERCENTILE NORMALIZATION
# ============================================================

# Using percentile ranks avoids combining incompatible
# raw units such as MIC, HC50, cosine similarity, etc.

# Lower predicted MIC = better
df["final_pct_potency"] = (
    df["apex_mean_mic_uM"]
    .rank(
        ascending=False,
        pct=True,
        method="average"
    )
)

# Higher predicted HC50 = better
df["final_pct_safety"] = (
    df["hemopi2_hc50_uM"]
    .rank(
        ascending=True,
        pct=True,
        method="average"
    )
)

# Higher APD6 realism diagnostic = better
df["final_pct_physchem"] = (
    df["APD6_physchem_realism_diagnostic"]
    .rank(
        ascending=True,
        pct=True,
        method="average"
    )
)

# Higher novelty = better
df["final_pct_novelty"] = (
    df["external_novelty_diagnostic"]
    .rank(
        ascending=True,
        pct=True,
        method="average"
    )
)

# Higher similarity to known potent AMP embedding references = better
df["final_pct_bio_embedding"] = (
    df["esm2_potent_mean_top5_cosine"]
    .rank(
        ascending=True,
        pct=True,
        method="average"
    )
)

# Higher PepSySco = more favorable
df["final_pct_synthesizability"] = (
    df["pepsysco_score"]
    .rank(
        ascending=True,
        pct=True,
        method="average"
    )
)


# ============================================================
# 6. TRANSPARENT MULTI-OBJECTIVE SCORE
# ============================================================

score_components = [
    "final_pct_potency",
    "final_pct_safety",
    "final_pct_physchem",
    "final_pct_novelty",
    "final_pct_bio_embedding",
    "final_pct_synthesizability"
]

# Equal contribution because public challenge weights
# are not disclosed.
df["final_multiobjective_score"] = (
    df[score_components]
    .mean(axis=1)
)

df["final_multiobjective_rank"] = (
    df["final_multiobjective_score"]
    .rank(
        ascending=False,
        method="first"
    )
    .astype(int)
)


# ============================================================
# 7. SAFETY / POTENCY AUXILIARY DIAGNOSTICS
# ============================================================

# This is predicted-model ratio only.
# It is NOT experimental selectivity.
df["final_predicted_HC50_to_MIC_ratio"] = (
    df["hemopi2_hc50_uM"]
    / df["apex_mean_mic_uM"]
)

# Published challenge-relevant potency threshold flag.
# Do NOT use as the sole ranking rule.
df["final_apex_mic_le_16"] = (
    df["apex_mean_mic_uM"] <= 16
)

# PepSySco published reference levels
df["final_pepsysco_ge_085"] = (
    df["pepsysco_score"] >= 0.85
)

df["final_pepsysco_ge_099"] = (
    df["pepsysco_score"] >= 0.99
)


# ============================================================
# 8. FINAL 50K LIBRARY
# ============================================================

library = (
    df.sort_values(
        [
            "final_multiobjective_score",
            "apex_mean_mic_uM",
            "hemopi2_hc50_uM",
            "external_novelty_diagnostic"
        ],
        ascending=[
            False,
            True,
            False,
            False
        ],
        kind="mergesort"
    )
    .head(LIBRARY_N)
    .copy()
)

library["final_library_rank"] = (
    np.arange(1, len(library) + 1)
)

assert len(library) == 50000
assert library["sequence"].nunique() == 50000


# ============================================================
# 9. LOCAL ALIGNMENT SIMILARITY FUNCTION
# ============================================================

# Smith-Waterman-style normalized local alignment similarity.
# For diversity pruning of Top100 only.

BLOSUM62 = substitution_matrices.load("BLOSUM62")

def local_alignment_similarity(seq1, seq2):

    # Self-scores for normalization
    self1 = pairwise2.align.localds(
        seq1,
        seq1,
        BLOSUM62,
        -10,
        -0.5,
        score_only=True
    )

    self2 = pairwise2.align.localds(
        seq2,
        seq2,
        BLOSUM62,
        -10,
        -0.5,
        score_only=True
    )

    pair = pairwise2.align.localds(
        seq1,
        seq2,
        BLOSUM62,
        -10,
        -0.5,
        score_only=True
    )

    denom = min(self1, self2)

    if denom <= 0:
        return 0.0

    return float(pair / denom)


# ============================================================
# 10. TOP100 SHORTLIST
# ============================================================

# Start from highest multi-objective candidates.
# A generous shortlist keeps greedy diversity pruning feasible.

SHORTLIST_N = 5000

shortlist = (
    library
    .sort_values(
        [
            "final_multiobjective_score",
            "apex_mean_mic_uM",
            "hemopi2_hc50_uM",
            "external_novelty_diagnostic"
        ],
        ascending=[
            False,
            True,
            False,
            False
        ]
    )
    .head(SHORTLIST_N)
    .copy()
)

print(
    "\nTop100 diversity pruning from shortlist:",
    len(shortlist)
)


# ============================================================
# 11. GREEDY TOP100 DIVERSITY SELECTION
# ============================================================

selected_rows = []
selected_sequences = []

checked_candidates = 0
rejected_similarity = 0

for idx, row in shortlist.iterrows():

    seq = row["sequence"]

    checked_candidates += 1

    keep_candidate = True
    max_similarity = 0.0

    for selected_seq in selected_sequences:

        sim = local_alignment_similarity(
            seq,
            selected_seq
        )

        max_similarity = max(
            max_similarity,
            sim
        )

        if sim > LOCAL_ALIGNMENT_SIM_THRESHOLD:

            keep_candidate = False
            rejected_similarity += 1
            break

    if keep_candidate:

        row_copy = row.copy()

        row_copy[
            "top100_max_similarity_to_earlier_selected"
        ] = max_similarity

        selected_rows.append(
            row_copy
        )

        selected_sequences.append(
            seq
        )

    if len(selected_rows) == TOP_N:
        break


if len(selected_rows) < TOP_N:
    raise RuntimeError(
        f"Only {len(selected_rows)} diverse peptides "
        f"were found from the {SHORTLIST_N} shortlist. "
        "Increase SHORTLIST_N before proceeding."
    )

top100 = pd.DataFrame(
    selected_rows
).reset_index(drop=True)

top100["final_top100_rank"] = (
    np.arange(1, len(top100) + 1)
)

assert len(top100) == 100
assert top100["sequence"].nunique() == 100


# ============================================================
# 12. VERIFY TOP100 INTERNAL LOCAL-ALIGNMENT DIVERSITY
# ============================================================

max_pairwise_similarity = 0.0
max_pair = None

for i in range(len(top100)):

    for j in range(i + 1, len(top100)):

        sim = local_alignment_similarity(
            top100.iloc[i]["sequence"],
            top100.iloc[j]["sequence"]
        )

        if sim > max_pairwise_similarity:

            max_pairwise_similarity = sim

            max_pair = (
                i,
                j
            )

print(
    "\nTop100 maximum internal local-alignment similarity:",
    max_pairwise_similarity
)

if max_pairwise_similarity > LOCAL_ALIGNMENT_SIM_THRESHOLD:

    raise RuntimeError(
        "Top100 diversity verification failed."
    )


# ============================================================
# 13. COMPLETE FULL CSV OUTPUTS
# ============================================================

library_full_csv = (
    OUT_DIR /
    "STEP12_FINAL_LIBRARY_50K_FULL.csv"
)

top100_full_csv = (
    OUT_DIR /
    "STEP12_FINAL_TOP100_FULL.csv"
)

library.to_csv(
    library_full_csv,
    index=False
)

top100.to_csv(
    top100_full_csv,
    index=False
)


# ============================================================
# 14. SUMMARY CSV COLUMNS
# ============================================================

summary_columns = [
    "sequence",
    "length",

    "final_multiobjective_score",

    "apex_mean_mic_uM",
    "hemopi2_hc50_uM",
    "final_predicted_HC50_to_MIC_ratio",

    "APD6_physchem_realism_diagnostic",
    "external_novelty_diagnostic",

    "esm2_potent_mean_top5_cosine",

    "pepsysco_score",

    "embedding_nn1_cosine_similarity",

    "final_pct_potency",
    "final_pct_safety",
    "final_pct_physchem",
    "final_pct_novelty",
    "final_pct_bio_embedding",
    "final_pct_synthesizability"
]


library_summary = (
    library[
        summary_columns +
        ["final_library_rank"]
    ]
    .copy()
)

top100_summary_cols = (
    summary_columns +
    [
        "final_top100_rank",
        "top100_max_similarity_to_earlier_selected"
    ]
)

top100_summary = (
    top100[
        top100_summary_cols
    ]
    .copy()
)


library_summary_csv = (
    OUT_DIR /
    "STEP12_FINAL_LIBRARY_50K_SUMMARY.csv"
)

top100_summary_csv = (
    OUT_DIR /
    "STEP12_FINAL_TOP100_SUMMARY.csv"
)

library_summary.to_csv(
    library_summary_csv,
    index=False
)

top100_summary.to_csv(
    top100_summary_csv,
    index=False
)


# ============================================================
# 15. FASTA WRITER
# ============================================================

def write_fasta(
    data,
    path,
    prefix,
    rank_column
):

    with open(
        path,
        "w"
    ) as f:

        for _, row in data.iterrows():

            rank = int(
                row[rank_column]
            )

            seq = row["sequence"]

            f.write(
                f">{prefix}_{rank:05d}\n"
            )

            f.write(
                f"{seq}\n"
            )


library_fasta = (
    OUT_DIR /
    "STEP12_FINAL_LIBRARY_50K.fasta"
)

top100_fasta = (
    OUT_DIR /
    "STEP12_FINAL_TOP100.fasta"
)

write_fasta(
    library,
    library_fasta,
    "AMP_LIBRARY",
    "final_library_rank"
)

write_fasta(
    top100,
    top100_fasta,
    "AMP_TOP100",
    "final_top100_rank"
)


# ============================================================
# 16. FINAL SUMMARY STATISTICS
# ============================================================

def summarize_pool(data, name):

    return {
        "pool": name,

        "n_sequences":
            int(len(data)),

        "unique_sequences":
            int(
                data["sequence"].nunique()
            ),

        "mean_apex_mic_uM":
            float(
                data["apex_mean_mic_uM"].mean()
            ),

        "median_apex_mic_uM":
            float(
                data["apex_mean_mic_uM"].median()
            ),

        "mean_hemopi2_hc50_uM":
            float(
                data["hemopi2_hc50_uM"].mean()
            ),

        "median_hemopi2_hc50_uM":
            float(
                data["hemopi2_hc50_uM"].median()
            ),

        "mean_predicted_HC50_to_MIC_ratio":
            float(
                data[
                    "final_predicted_HC50_to_MIC_ratio"
                ].mean()
            ),

        "mean_external_novelty":
            float(
                data[
                    "external_novelty_diagnostic"
                ].mean()
            ),

        "mean_biological_embedding_similarity":
            float(
                data[
                    "esm2_potent_mean_top5_cosine"
                ].mean()
            ),

        "mean_pepsysco":
            float(
                data[
                    "pepsysco_score"
                ].mean()
            ),

        "median_pepsysco":
            float(
                data[
                    "pepsysco_score"
                ].median()
            ),

        "mean_final_multiobjective_score":
            float(
                data[
                    "final_multiobjective_score"
                ].mean()
            ),

        "n_pepsysco_ge_085":
            int(
                (
                    data["pepsysco_score"]
                    >= 0.85
                ).sum()
            ),

        "n_pepsysco_ge_099":
            int(
                (
                    data["pepsysco_score"]
                    >= 0.99
                ).sum()
            )
    }


summary_stats = pd.DataFrame(
    [
        summarize_pool(
            library,
            "Final Library 50K"
        ),

        summarize_pool(
            top100,
            "Final Top100"
        )
    ]
)

summary_stats_path = (
    OUT_DIR /
    "STEP12_FINAL_SELECTION_SUMMARY_STATS.csv"
)

summary_stats.to_csv(
    summary_stats_path,
    index=False
)


# ============================================================
# 17. SHA256
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(block)

    return h.hexdigest()


# ============================================================
# 18. MANIFEST
# ============================================================

manifest = {

    "step": "12",

    "method":
        "transparent equal-percentile multi-objective ranking "
        "+ greedy local-alignment diversity pruning",

    "input_rows":
        int(len(df)),

    "input_unique_sequences":
        int(
            df["sequence"].nunique()
        ),

    "final_library_n":
        int(len(library)),

    "final_top100_n":
        int(len(top100)),

    "score_components": {
        "potency":
            "APEX mean predicted MIC; lower is better",

        "safety":
            "HemoPI2 predicted HC50; higher is better",

        "physchem":
            "APD6 physicochemical realism diagnostic",

        "novelty":
            "external novelty diagnostic",

        "biological_embedding":
            "ESM2 potent-reference mean Top-5 cosine",

        "synthesizability":
            "PepSySco"
    },

    "component_weighting":
        "equal percentile contribution",

    "official_hidden_weights_claimed":
        False,

    "top100_internal_diversity": {
        "method":
            "normalized BLOSUM62 local alignment",

        "threshold":
            LOCAL_ALIGNMENT_SIM_THRESHOLD,

        "selection":
            "greedy",

        "maximum_verified_similarity":
            float(
                max_pairwise_similarity
            )
    },

    "random_seed":
        RANDOM_SEED,

    "hard_filter_50k":
        False,

    "outputs": {

        "library_full_csv":
            str(library_full_csv),

        "top100_full_csv":
            str(top100_full_csv),

        "library_summary_csv":
            str(library_summary_csv),

        "top100_summary_csv":
            str(top100_summary_csv),

        "library_fasta":
            str(library_fasta),

        "top100_fasta":
            str(top100_fasta),

        "summary_stats":
            str(summary_stats_path)
    },

    "sha256": {
        "library_full_csv":
            sha256_file(
                library_full_csv
            ),

        "top100_full_csv":
            sha256_file(
                top100_full_csv
            ),

        "library_fasta":
            sha256_file(
                library_fasta
            ),

        "top100_fasta":
            sha256_file(
                top100_fasta
            )
    }
}


manifest_path = (
    OUT_DIR /
    "STEP12_SELECTION_MANIFEST.json"
)

with open(
    manifest_path,
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ============================================================
# 19. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("STEP 12 FINAL VALIDATION")
print("=" * 80)

print(
    "Input:",
    len(df),
    "rows |",
    df["sequence"].nunique(),
    "unique"
)

print(
    "\nFinal Library:",
    len(library),
    "| unique:",
    library["sequence"].nunique()
)

print(
    "Final Top100:",
    len(top100),
    "| unique:",
    top100["sequence"].nunique()
)

print(
    "\nTop100 max internal local-alignment similarity:",
    round(
        max_pairwise_similarity,
        6
    )
)

print(
    "Top100 candidates checked:",
    checked_candidates
)

print(
    "Candidates rejected by diversity:",
    rejected_similarity
)

print("\nSummary statistics:")
display(summary_stats)

print("\nTop 10 final Top100:")
display(
    top100_summary.head(10)
)


# ============================================================
# 20. ZIP ALL OUTPUTS
# ============================================================

import shutil

zip_path = shutil.make_archive(
    "/content/STEP12_FINAL_SELECTION_OUTPUTS",
    "zip",
    OUT_DIR
)

print("\nZIP:", zip_path)


# ============================================================
# 21. DOWNLOAD ZIP
# ============================================================

files.download(
    zip_path
)

print("\nDONE")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.2 MB/s eta 0:00:00
Upload:
STEP11_60K_WITH_DIVERSITY_DIAGNOSTICS.csv


/usr/local/lib/python3.13/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


Saving STEP11_60K_WITH_DIVERSITY_DIAGNOSTICS.csv to STEP11_60K_WITH_DIVERSITY_DIAGNOSTICS.csv

Input file: STEP11_60K_WITH_DIVERSITY_DIAGNOSTICS.csv
Rows: 60000
Unique sequences: 60000

All required scoring columns found.
apex_mean_mic_uM: missing = 0
hemopi2_hc50_uM: missing = 0
APD6_physchem_realism_diagnostic: missing = 0
external_novelty_diagnostic: missing = 0
esm2_potent_mean_top5_cosine: missing = 0
pepsysco_score: missing = 0
embedding_nn1_cosine_similarity: missing = 0

Top100 diversity pruning from shortlist: 5000

Top100 maximum internal local-alignment similarity: 0.39855072463768115

STEP 12 FINAL VALIDATION
Input: 60000 rows | 60000 unique

Final Library: 50000 | unique: 50000
Final Top100: 100 | unique: 100

Top100 max internal local-alignment similarity: 0.398551
Top100 candidates checked: 139
Candidates rejected by diversity: 39

Summary statistics:


,pool,n_sequences,unique_sequences,mean_apex_mic_uM,median_apex_mic_uM,mean_hemopi2_hc50_uM,median_hemopi2_hc50_uM,mean_predicted_HC50_to_MIC_ratio,mean_external_novelty,mean_biological_embedding_similarity,mean_pepsysco,median_pepsysco,mean_final_multiobjective_score,n_pepsysco_ge_085,n_pepsysco_ge_099
0,Final Library 50K,50000,50000,176.079915,176.357209,39.498859,29.3665,0.239884,47.567069,0.923953,0.863602,0.88141,0.539498,31852,231
1,Final Top100,100,100,117.945195,114.511215,71.672130,67.8340,0.638675,53.609670,0.963155,0.923331,0.92781,0.812175,97,1



Top 10 final Top100:


,sequence,length,final_multiobjective_score,apex_mean_mic_uM,hemopi2_hc50_uM,final_predicted_HC50_to_MIC_ratio,APD6_physchem_realism_diagnostic,external_novelty_diagnostic,esm2_potent_mean_top5_cosine,pepsysco_score,embedding_nn1_cosine_similarity,final_pct_potency,final_pct_safety,final_pct_physchem,final_pct_novelty,final_pct_bio_embedding,final_pct_synthesizability,final_top100_rank,top100_max_similarity_to_earlier_selected
0,FWKLRWICRC,10,0.874076,115.290987,71.995,0.624463,90.457,53.333,0.949488,0.99934,0.993058,0.842733,0.890800,0.905750,0.857325,0.748250,0.999600,1,0.000000
1,VRIQRYWKITLGLKKAMW,18,0.872457,86.184671,107.815,1.250977,85.680,55.556,0.972724,0.91639,0.994601,0.947050,0.970617,0.664275,0.924133,0.989200,0.739467,2,0.304348
2,TLWWRWKSYRLKLLNKLADTAA,22,0.872150,83.766942,73.849,0.881601,91.246,52.632,0.962928,0.91359,0.988271,0.954050,0.896250,0.932217,0.819692,0.905767,0.724925,3,0.253623
3,SWLELKFYRWIKKLIKQQT,19,0.857876,111.769242,68.365,0.611662,89.524,50.000,0.972043,0.94579,0.991153,0.856883,0.879650,0.870867,0.676917,0.986117,0.876825,4,0.289855
4,CARFATYKWRGVLKELKKIM,20,0.853015,124.300120,98.535,0.792718,89.298,54.839,0.953620,0.92781,0.993483,0.803483,0.956933,0.861142,0.906083,0.795133,0.795317,5,0.221154
5,RWHIFYRFLLWK,12,0.850532,93.373938,35.296,0.378007,88.756,51.515,0.967521,0.95872,0.993020,0.925317,0.679633,0.836725,0.775667,0.953750,0.932100,6,0.276316
6,FWRYRYFYQIKKKNLLLLAA,20,0.844553,118.589549,32.640,0.275235,89.625,57.576,0.971285,0.92607,0.989870,0.828567,0.636692,0.874567,0.958908,0.982450,0.786133,7,0.362319
7,WNASKVYGLYFRRVMLKIKKWA,22,0.842163,91.320934,66.879,0.732351,87.928,52.174,0.963113,0.91718,0.992798,0.931783,0.874175,0.795600,0.800917,0.907717,0.742783,8,0.210526
8,MTWNNVKLKWWASILKQYINKKW,23,0.839219,128.715778,59.474,0.462057,88.648,52.941,0.966354,0.92691,0.991351,0.784017,0.846550,0.831508,0.839825,0.942383,0.791033,9,0.318841
9,MAWKNIGLNKLNKLMASILNIKKKL,25,0.834614,100.097620,94.778,0.946856,87.130,51.111,0.969103,0.90958,0.989191,0.901883,0.949800,0.754050,0.727992,0.967600,0.706358,10,0.296000



ZIP: /content/STEP12_FINAL_SELECTION_OUTPUTS.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DONE
